In [1]:
import os
from getpass import getpass
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
import random

from transformers import AutoTokenizer

from sklearn.model_selection import train_test_split

SEED = 42


In [2]:
# Check to see if files are downloaded before running the following cell
!ls -la data

ls: cannot access 'data': No such file or directory


In [3]:
# Get Kaggle API Token
os.environ['KAGGLE_API_TOKEN'] = getpass("Kaggle API token: ")

# Download kaggle competition datasets
!kaggle competitions download -c llm-classification-finetuning

# Unzip the files
!unzip -q llm-classification-finetuning.zip -d data

# Lists the files
!ls -la data

100% 57.0M/57.0M [00:00<00:00, 131MB/s] 

total 179884
drwxr-xr-x 2 root root      4096 Jul 11 18:14 .
drwxr-xr-x 1 root root      4096 Jul 11 18:14 ..
-rw-r--r-- 1 root root       237 Oct  9  2024 sample_submission.csv
-rw-r--r-- 1 root root     10658 Oct  9  2024 test.csv
-rw-r--r-- 1 root root 184175071 Oct  9  2024 train.csv


# Archetecure of model:

### input variables
```
prompt = prompt
response_1 = either reponse_a or response_b
response_2 = the other response_a/b
```
### Encoding model
```
model_1 = DeBERTA v3 base
```

### Classification model
```
model_2 = classification model
```
### pipeline of data
```
prompt -> model_1 -> output_p
response_1 -> model_1 -> output_1
response_2 -> model_1 -> output_2

(output_p, output_1, output_2) -> model_2 -> (prob of first response, prob of second response, prob of tie)
```



In [4]:
class LLMModelsPref(Dataset):
    """
    A custom Dataset class for handling prompts, responses from two models, and preference labels.
    This Dataset uses lazy loading.
    """
    def __init__(self, prompts, responses_a, responses_b, labels, tokenizer, max_len=512, prob=0.5):
        """
        Initializes the dataset.

        Args:
            prompts (list): A list of strings containing the prompts.
            responses_a (list): A list of strings containing the responses to
                model_a.
            responses_b (list): A list of strings containing the responses to
                model_b.
            labels (ndarray): A 1-d array of integers containing the preferred
                response.
                - 0: response_a preferred
                - 1: response_b preferred
                - 2: tie
            tokenizer (object): An object used to convert text into numerical
                indices.
            max_len (int): The maximum sequence length allowed for each
                prompt/response.
            prob (float): The probability (between 0 and 1) of permuting the
                order of responses.
        """
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.prob = prob
        self.prompts = prompts
        self.responses_a = responses_a
        self.responses_b = responses_b
        self.labels = labels

    def __len__(self):
        """
        Returns the number of samples in the dataset.

        Returns:
            int: The number of samples in the dataset.
        """
        return len(self.prompts)

    def __getitem__(self, idx):
        """
        Retrieves and tokenizes one sample, randomly swaps response_a/b (with
            probability self.prob) to reduce positional bias in training.

        Args:
            idx (int): Index of the sample to retrieve.

        Returns:
            dict: A dictionary containing:
                - prompt (Tensor): Token IDs for the prompt.
                - prompt_mask (Tensor): Attention mask for the prompt.
                - response_a (Tensor): Token IDs for response_a (possibly
                    swapped).
                - response_a_mask (Tensor): Attention mask for response_a.
                - response_b (Tensor): Token IDs for response_b (possibly
                    swapped).
                - respons_b_mask (Tensor): Attention mask for response_b.
                - label (Tensor): Integer class label 0/1/2,
                    adjusted if swapped.
        """
        prompt = self.prompts[idx]
        label = self.labels[idx]
        response_a = self.responses_a[idx]
        response_b = self.responses_b[idx]

        # Swap responses and adjust label with probability self.prob.
        if random.random() < self.prob:
            response_a, response_b = response_b, response_a
            if label == 0:
                label = 1
            elif label ==1:
                label = 0

        encoded_p = self.tokenizer(prompt, max_length=self.max_len,
                                   truncation=True, padding="max_length",
                                   return_tensors="pt")
        encoded_a = self.tokenizer(response_a, max_length=self.max_len,
                                   truncation=True, padding="max_length",
                                   return_tensors="pt")
        encoded_b = self.tokenizer(response_b, max_length=self.max_len,
                                   truncation=True, padding="max_length",
                                   return_tensors="pt")

        # After retrieving the tokens, we remove the extra dimension that was added
        # by the tokenizer (the batch dim) even for a single string.
        prompt = encoded_p["input_ids"].squeeze(0)
        response_a = encoded_a["input_ids"].squeeze(0)
        response_b = encoded_b["input_ids"].squeeze(0)

        prompt_mask = encoded_p["attention_mask"].squeeze(0)
        response_a_mask = encoded_a["attention_mask"].squeeze(0)
        response_b_mask = encoded_b["attention_mask"].squeeze(0)

        label = torch.tensor(label, dtype=torch.long)

        return {
            "prompt": prompt, "prompt_mask": prompt_mask,
            "response_a": response_a, "response_a_mask": response_a_mask,
            "response_b": response_b, "response_b_mask": response_b_mask,
            "label": label
        }




In [5]:

df = pd.read_csv("data/train.csv")
#df = df.head(200)

labels_one_hot = df[["winner_model_a","winner_model_b","winner_tie"]]
labels = labels_one_hot.values.argmax(axis = 1)


In [ ]:


tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [7]:
train_df, val_df, train_labels, val_labels = train_test_split(
    df,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=SEED
)

val_df, test_df, val_labels, test_labels = train_test_split(
    val_df,
    val_labels,
    test_size=0.5,
    stratify=val_labels,
    random_state=SEED
)

In [9]:
train_dataset = LLMModelsPref(
    prompts = train_df["prompt"].tolist(),
    responses_a = train_df["response_a"].tolist(),
    responses_b = train_df["response_b"].tolist(),
    labels= train_labels,
    tokenizer = tokenizer
)
val_dataset = LLMModelsPref(
    prompts = val_df["prompt"].tolist(),
    responses_a = val_df["response_a"].tolist(),
    responses_b = val_df["response_b"].tolist(),
    labels= val_labels,
    tokenizer = tokenizer,
    prob=0
)
test_dataset = LLMModelsPref(
    prompts = test_df["prompt"].tolist(),
    responses_a = test_df["response_a"].tolist(),
    responses_b = test_df["response_b"].tolist(),
    labels= test_labels,
    tokenizer = tokenizer,
    prob=0
)

In [10]:
batch_size = 32
train_loader = DataLoader( train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader( val_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader( test_dataset, batch_size=batch_size, shuffle=True)